# Práctica de laboratorio — Limpieza y preprocesamiento programático de la **Encuesta COVID** con Python

En esta práctica trabajarás con un conjunto de datos real procedente del cuestionario **EncuestaCOVID.docx**.  
Tu trabajo **no** consiste en limpiar el fichero a mano en Excel, sino en hacerlo **exclusivamente con código Python** sobre tus **120 filas asignadas**.

## Idea general de la práctica

Debes trabajar **columna a columna** (o, cuando proceda, **grupo de columnas de una misma pregunta**) y hacer lo siguiente:

1. **Identificar** a qué pregunta del cuestionario corresponde cada columna.
2. **Analizar** el tipo de dato real que contiene esa columna.
3. **Detectar problemas**: valores faltantes, espacios, errores tipográficos, codificaciones inconsistentes, categorías redundantes, outliers, mezclas de texto y número, etc.
4. **Proponer y escribir código** que limpie la columna o el grupo de columnas.
5. **Justificar** cada decisión de limpieza.
6. **Construir** una versión limpia del dataset y una versión reducida para análisis/modelado.

## Técnicas que debes usar a lo largo de la práctica

A partir de las transparencias de preprocesamiento, debes aplicar muchas de estas técnicas cuando tengan sentido:

- detección de valores faltantes;
- eliminación o imputación justificada;
- detección de inconsistencias;
- normalización de categorías de texto;
- detección de duplicados;
- detección y tratamiento de outliers;
- creación de variables derivadas;
- codificación de variables categóricas;
- escalado/normalización;
- discretización;
- reducción de variables.

## Importante

- **No modifiques el archivo XLSX manualmente**.
- Mantén siempre una copia cruda: `df_raw`.
- Toda limpieza debe quedar reflejada en código.
- No sobrescribas una columna original sin haberla inspeccionado antes.
- Cuando crees una versión limpia, usa nombres como `*_clean`, `*_num`, `*_ord`, etc., o bien documenta claramente el reemplazo.

## Entregables

Al final deberás entregar, como mínimo:

1. Este notebook completado y ejecutado.
2. Un dataframe limpio `df_limpio`.
3. Un dataframe reducido `df_modelo` con aproximadamente **15 variables útiles**.
4. Una **tabla de auditoría** donde se vea, para cada columna original:
   - a qué pregunta pertenece,
   - qué problema(s) detectaste,
   - qué estrategia aplicaste,
   - qué columna(s) final(es) generaste.

# 0. Carga de librerías y selección de tus 120 filas

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option('display.max_rows', None)
pd.set_option("display.max_colwidth", 120)

In [2]:
# Carpeta de trabajo
carpeta = "/home/miguel/Desktop/uma-25-26-2/AC1/Practica1"
print("Carpeta de trabajo:", carpeta)

# Descarga en esta carpeta, desde el Campus Virtual, al menos estos ficheros:
# - DatosEncuestaCOVID.xlsx
# - EncuestaCOVID.docx
# - CodigosEstudiantado2025-2026.txt

Carpeta de trabajo: /home/miguel/Desktop/uma-25-26-2/AC1/Practica1


In [2]:
codigo = 40  # TODO: sustituye 69 por tu código de estudiante

inicio = 1 + (codigo - 1) * 120
fin = 120 + (codigo - 1) * 120

print(f"Tus filas van desde {inicio} hasta {fin}")

Tus filas van desde 4681 hasta 4800


In [3]:
df_raw = pd.read_excel("DatosEncuestaCOVID.xlsx")
print("Número total de filas del XLSX:", len(df_raw))

df = df_raw.iloc[inicio - 1 : fin].copy()
print("Número de filas de tu subconjunto:", len(df))

ImportError: Missing optional dependency 'openpyxl'.  Use pip or conda to install openpyxl.

# 1. Exploración inicial obligatoria

**Antes de limpiar nada**, responde con código a estas preguntas:

- ¿Cuántas filas tiene tu subconjunto?
- ¿Cuántas columnas tiene?
- ¿Qué tipos de datos detecta pandas?
- ¿Qué columnas parecen numéricas pero no lo son?
- ¿Qué columnas parecen categóricas?
- ¿Qué columnas pertenecen a preguntas multirrespuesta?
- ¿Hay alguna discrepancia entre el número de columnas esperado y el número de columnas real?

In [5]:
print(f"El dataset tiene {df.shape[0]} filas y {df.shape[1]} columnas.")

El dataset tiene 120 filas y 153 columnas.


In [6]:
df.info()
df.head()
print(f"El dataset presenta datos de tipo float, object y string.")
print(f"Columnas como 'Convivencia-antes_1' parece numérica pero no lo es.")
print(f"Columnas como el 'Sexo_1' son categóricas.")
print(f"Columnas como el 'Transporte' son multirespuesta.")
print(f"En principio solo eran 53 preguntas, por lo que debería haber 53 columnas pero nos encontramos con 153 columnas.")

<class 'pandas.DataFrame'>
RangeIndex: 120 entries, 4680 to 4799
Columns: 153 entries, 1.CPBARRIO-ANTES_1 to 51.COVID-ANTICUERPOS_1
dtypes: float64(5), object(7), str(141)
memory usage: 143.6+ KB
El dataset presenta datos de tipo float, object y string.
Columnas como 'Convivencia-antes_1' parece numérica pero no lo es.
Columnas como el 'Sexo_1' son categóricas.
Columnas como el 'Transporte' son multirespuesta.
En principio solo eran 53 preguntas, por lo que debería haber 53 columnas pero nos encontramos con 153 columnas.


In [7]:
df.columns

Index(['1.CPBARRIO-ANTES_1', '1.CPBARRIO-ANTES_2', '2.SEXO_1', '3.EDAD_1',
       '4.PESO_1', '5TALLA_1', '6.DOMICILIO-ANTES_1', '7.AFLUENCIA-ANTES_1',
       '8.CONVIVENCIA-ANTES_1', '9.ED,DES-,NTES_1',
       ...
       '47.VACUNAS_5', '47.VACUNAS_6', '47.VACUNAS_7', '47.VACUNAS_8',
       '47.VACUNAS_9', '47.VACUNAS_10', '48.COVID-CONTACTO_1',
       '49.COVID-SOSPECHA_1', '50.COVID-PCR_1', '51.COVID-ANTICUERPOS_1'],
      dtype='str', length=153)

# 2. Funciones auxiliares para la auditoría

Las siguientes funciones **no resuelven** la práctica, pero te ayudan a inspeccionar el dataset de forma sistemática.

Úsalas tantas veces como necesites.

In [34]:
def resumen_columna(df, col, top=15):
    s = df[col]
    out = pd.DataFrame({
        "columna": [col],
        "dtype": [s.dtype],
        "n": [len(s)],
        "n_missing": [s.isna().sum()],
        "pct_missing": [100 * s.isna().mean()],
        "n_unicos": [s.nunique(dropna=True)]
    })
    display(out)
    print("\nPrimeros valores no nulos:")
    display(s.dropna().astype(str).head(top))
    print("\nFrecuencias (incluyendo NA):")
    display(s.astype("object").value_counts(dropna=False).head(top))


def ver_numerica(df, col, bins=20):
    s = pd.to_numeric(df[col], errors="coerce")
    display(s.describe())
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    s.plot(kind="hist", bins=bins, ax=ax[0], title=f"Histograma: {col}")
    s.plot(kind="box", ax=ax[1], title=f"Boxplot: {col}")
    plt.tight_layout()
    plt.show()


def ver_categorica(df, col, top=20):
    s = df[col].astype("object")
    display(s.value_counts(dropna=False).head(top))


def columnas_con_prefijo(df, pregunta):
    pregunta = str(pregunta)
    patron = rf"^{re.escape(pregunta)}(?:\.|_|[A-Z])"
    return [c for c in df.columns if re.search(patron, c)]


def columnas_por_bloque(df, preguntas):
    cols = []
    for p in preguntas:
        cols.extend(columnas_con_prefijo(df, p))
    # preserva el orden original del dataframe
    cols = [c for c in df.columns if c in cols]
    return cols

# 3. Normalización inicial de valores faltantes y espacios

Antes de estudiar cada columna, crea una copia de trabajo y **normaliza representaciones obvias** de datos faltantes.

No conviertas todavía a número ni recodifiques categorías complejas: primero deja homogéneo el dataset.

In [9]:
df_trabajo = df.copy()

# TODO:
# 1) Convierte strings vacíos y espacios a NaN.
# 2) Busca otras representaciones de faltantes: "NA", "N/A", "No consta", "-", etc.
# 3) Decide cuáles debes convertir a NaN y cuáles no.
#
# Ejemplo orientativo:
# df_trabajo = df_trabajo.replace(r"^\s*$", np.nan, regex=True)

df_trabajo = df_trabajo.replace([r"^\s*$", "-", "No sé, no me consta", "No se, no me consta", ",", "NA", "N/A"], np.nan, regex=True)
df_trabajo.head()

,1.CPBARRIO-ANTES_1,1.CPBARRIO-ANTES_2,2.SEXO_1,3.EDAD_1,4.PESO_1,5TALLA_1,6.DOMICILIO-ANTES_1,7.AFLUENCIA-ANTES_1,8.CONVIVENCIA-ANTES_1,"9.ED,DES-,NTES_1",10.PROFESION-ANTES_1,11.LUGARTRABAJO-ANTES_1,12.OCIO_1,13.TRANSPORTE_1,13.TRANSPORTE_2,13.TRANSPORTE_3,13.TRANSPORTE_4,13.TRANSPORTE_5,13.TRANSPORTE_6,13.TRANSPORTE_7,14.EXTRANJERO_1,15.DESPLAZAMIENTO-NACIONAL_1,16.CPBARRIO-DURANTE_1,16.CPBARRIO-DURANTE_2,17.DOMICILIO-DURANTE_1,18.AFLUENCIA-DURANTE_1,19.CONVIVENCIA-DURANTE_1,20.EDADES-DURANTE_1,21.MASCOTAS_1,21.MASCOTAS_2,21.MASCOTAS_3,21.MASCOTAS_4,21.MASCOTAS_5,22.PROFESION-DURANTE_1,22.PROFESION-DURANTE_2,22.PROFESION-DURANTE_3,22.PROFESION-DURANTE_4,22.PROFESION-DURANTE_5,22.PROFESION-DURANTE_6,22.PROFESION-DURANTE_7,22.PROFESION-DURANTE_8,22.PROFESION-DURANTE_9,22.PROFESION-DURANTE_10,22.PROFESION-DURANTE_11,22.PROFESION-DURANTE_12,23.LUGARTRABAJO-DURANTE_1,24.SEGURIDAD-MES_1,24.SEGURIDAD-MES_2,24.SEGURIDAD-MES_3,24.SEGURIDAD-MES_4,25.SEGURIDAD-FRECUENCIA_1,25.SEGURIDAD-FRECUENCIA_2,25.SEGURIDAD-FRECUENCIA_3,25.SEGURIDAD-FRECUENCIA_4,25.SEGURIDAD-FRECUENCIA_5,26.FUMADOR_1,27.ALCOHOL_1,28.BEBIDAS_1,28.BEBIDAS_2,28.BEBIDAS_3,28.BEBIDAS_4,28.BEBIDAS_5,28.BEBIDAS_6,29.ANTIDEPRESIVOS_1,30.TRANQUILIZANTES_1,31.TRATAMIENTO-DOLOR_1,32.SUPLEMENTOS_1,32.SUPLEMENTOS_2,32.SUPLEMENTOS_3,33.ESTADO-SALUD_1,34.ENF-METABOLICA_1,34.ENF-METABOLICA_2,34.ENF-METABOLICA_3,34.ENF-METABOLICA_4,34.ENF-METABOLICA_5,34.ENF-METABOLICA_6,34.ENF-METABOLICA_7,34.ENF-METABOLICA_8,34.ENF-METABOLICA_9,35.ENF-AUTOINMUNE_1,35.ENF-AUTOINMUNE_2,35.ENF-AUTOINMUNE_3,35.ENF-AUTOINMUNE_4,35.ENF-AUTOINMUNE_5,35.ENF-AUTOINMUNE_6,35.ENF-AUTOINMUNE_7,35.ENF-AUTOINMUNE_8,36.ENF-ALERGIA_1,36.ENF-ALERGIA_2,36.ENF-ALERGIA_3,36.ENF-ALERGIA_4,36.ENF-ALERGIA_5,36.ENF-ALERGIA_6,36.ENF-ALERGIA_7,37.ENF-RESPIRATORIA_1,37.ENF-RESPIRATORIA_2,37.ENF-RESPIRATORIA_3,37.ENF-RESPIRATORIA_4,37.ENF-RESPIRATORIA_5,37.ENF-RESPIRATORIA_6,37.ENF-RESPIRATORIA_7,38.ENF-CARDIACA_1,38.ENF-CARDIACA_2,38.ENF-CARDIACA_3,38.ENF-CARDIACA_4,38.ENF-CARDIACA_5,38.ENF-CARDIACA_6,39.ENF-OTRAS_1,39.ENF-OTRAS_2,39.ENF-OTRAS_3,39.ENF-OTRAS_4,39.ENF-OTRAS_5,39.ENF-OTRAS_6,39.ENF-OTRAS_7,40.EPOC-RIESGO_1,40.EPOC-RIESGO_2,41.SALUD-TRIMESTRE_1,42.HOSPITAL-INGRESO_1,43.HOSPITAL-PRUEBAS_1,44.PATOLOGIA-CONSIS_1,45.PATOLOGIA-SINASIS_1,46.SINTOMATOLOGIA_1,46.SINTOMATOLOGIA_2,46.SINTOMATOLOGIA_3,46.SINTOMATOLOGIA_4,46.SINTOMATOLOGIA_5,46.SINTOMATOLOGIA_6,46.SINTOMATOLOGIA_7,46.SINTOMATOLOGIA_8,46.SINTOMATOLOGIA_9,46.SINTOMATOLOGIA_10,46.SINTOMATOLOGIA_11,46.SINTOMATOLOGIA_12,46.SINTOMATOLOGIA_13,46.SINTOMATOLOGIA_14,46.SINTOMATOLOGIA_15,46.SINTOMATOLOGIA_16,46.SINTOMATOLOGIA_17,46.SINTOMATOLOGIA_18,47.VACUNAS_1,47.VACUNAS_2,47.VACUNAS_3,47.VACUNAS_4,47.VACUNAS_5,47.VACUNAS_6,47.VACUNAS_7,47.VACUNAS_8,47.VACUNAS_9,47.VACUNAS_10,48.COVID-CONTACTO_1,49.COVID-SOSPECHA_1,50.COVID-PCR_1,51.COVID-ANTICUERPOS_1
4680,14005,14005,Hombre:,42,72.0,167,Piso en bloque de menos 50 viviendas,NaN,0,0,Ámbito educativo,Instituto de Enseñanza Secundaria,En lugares abiertos y cerrados por igual,Frecuente,Baja,Baja,Muy baja,Baja,NaN,Frecuente,NO,NaN,14005.0,Ciudad Jardín,Piso en bloque de más de 50 viviendas,NaN,0,0,Gato,NaN,NaN,NaN,NaN,NaN,Mediante teletrabajo,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,En el hogar (teletrabajo/en paro/ tareas domésticas/confinado),Marzo,NaN,NaN,NaN,Frecuentemente,Algunas veces,Muy frecuentemente,Frecuentemente,Muy frecuentemente,No,NaN,Habitualmente,Habitualmente,De vez en cuando,No consumo,NaN,NaN,No,No,No,De vez en cuando,Habitualmente,Habitualmente,Excelente,No,No,No,Si,No,No,No,NaN,NaN,No,No,No,No,No,No,NaN,NaN,No,No,No,No,No,No,NaN,No,No,Si,No,No,NaN,NaN,No,No,No,No,NaN,NaN,No,No,Si,No,No,NaN,NaN,No,No,Excelente,No,No,NaN,No,No,No,Si,No,Si,No,No,No,No,No,No,No,No,No,Si,No,No,NaN,Si,No,NaN,NaN,NaN,NaN,NaN,No,No,NaN,No,No,No,no
4681,28002,Prosperidad,Mujer:,56,65.0,167,Piso en bloque de menos 50 viviendas,Medio (movimiento de residentes),0,NaN,Medios de comunicación,NaN,NaN,NaN,NaN,Baja,Baja,Frecuent

In [10]:
# TODO: crea una tabla rápida con número y porcentaje de faltantes por columna
# missing = ...
# display(missing.head(30))

missing = pd.DataFrame({
    "nulos" : df_trabajo.isnull().sum(),
    "porcentaje" : (df_trabajo.isnull().sum() / len(df_trabajo)) * 100
})

display(missing.head(30))

,nulos,porcentaje
1.CPBARRIO-ANTES_1,1,0.833333
1.CPBARRIO-ANTES_2,12,10.000000
2.SEXO_1,1,0.833333
3.EDAD_1,0,0.000000
4.PESO_1,0,0.000000
5TALLA_1,0,0.000000
6.DOMICILIO-ANTES_1,0,0.000000
7.AFLUENCIA-ANTES_1,41,34.166667
8.CONVIVENCIA-ANTES_1,0,0.000000
"9.ED,DES-,NTES_1",80,66.666667


# 4. Diccionario del cuestionario y tabla maestra de auditoría

En el fichero **EncuestaCOVID.docx** aparecen las preguntas del cuestionario.  
A continuación tienes un **diccionario de preguntas** resumido para ayudarte a relacionar cada columna del XLSX con la pregunta correspondiente.

**Tu tarea no es editar este diccionario**, sino usarlo para construir una auditoría **columna a columna**.

In [11]:
diccionario_preguntas = pd.DataFrame([('P1', '1', 'Código postal antes del confinamiento', 'texto/código', 'longitud, dígitos, ceros a la izquierda, faltantes'), ('P1', '1.1', 'Barrio antes del confinamiento', 'texto nominal', 'espacios, mayúsculas, tildes, abreviaturas, duplicados semánticos'), ('P1', '2', 'Sexo', 'categórica nominal', 'H/M, Hombre/Mujer, mayúsculas, categorías extrañas'), ('P1', '3', 'Edad', 'numérica', 'conversión a número, valores imposibles, outliers, imputación'), ('P1', '4', 'Peso (kg)', 'numérica', 'conversión a número, unidades, extremos, ceros/imposibles'), ('P1', '5', 'Talla (cm)', 'numérica', 'conversión a número, cm frente a m, extremos, ceros/imposibles'), ('P1', '6', 'Tipo de domicilio antes', 'categórica nominal', 'unificar categorías, otros, faltantes'), ('P1', '7', 'Afluencia en la zona antes', 'categórica ordinal', 'bajo/medio/alto, orden lógico, variantes de texto'), ('P1', '8', 'Número de convivientes antes', 'numérica discreta', 'enteros, negativos, extremos, faltantes'), ('P1', '9', 'Edades de convivientes antes', 'texto/lista', 'listas mal formadas, separadores, extracción de resúmenes'), ('P1', '10', 'Ámbito profesional antes', 'categórica nominal', 'sinónimos, otros, categorías mezcladas'), ('P1', '11', 'Descripción del lugar de trabajo antes', 'categórica nominal', 'unificación y categorías raras'), ('P1', '12', 'Tiempo libre antes', 'categórica nominal', "unificación y categoría 'Otro'"), ('P1', '13', 'Medio de transporte habitual antes', 'tabla/orden de preferencias', 'listas, separadores, categorías redundantes'), ('P1', '14', 'Viaje al extranjero', 'mixta sí/no + texto', 'separar indicador y destino, faltantes'), ('P1', '15', 'Desplazamiento dentro de España', 'categórica nominal', 'sí/no/madrid/barcelona/otra/otro'), ('P2', '16', 'Código postal durante el confinamiento', 'texto/código', 'igual que P1.1'), ('P2', '16.1', 'Barrio durante el confinamiento', 'texto nominal', 'igual que P1.1'), ('P2', '17', 'Tipo de domicilio durante', 'categórica nominal', 'comparar con P1.6'), ('P2', '18', 'Afluencia en la zona durante', 'categórica ordinal', 'comparar con P1.7'), ('P2', '19', 'Número de convivientes durante', 'numérica discreta', 'enteros, extremos, comparación con P1.8'), ('P2', '20', 'Edades de convivientes durante', 'texto/lista', 'parseo, resúmenes, consistencia con P2.19'), ('P2', '21', 'Convivencia con mascotas', 'multirrespuesta / nominal', 'no/perro/gato/otras; consistencia entre marcas'), ('P2', '22', 'Ámbito y forma de ejercer la profesión durante', 'tabla multirrespuesta', 'una respuesta por fila, categorías válidas'), ('P2', '23', 'Descripción del lugar de trabajo durante', 'categórica nominal', 'unificación y coherencia'), ('P2', '24', 'Mes de inicio de medidas de seguridad', 'categórica ordinal', 'enero-febrero-marzo-abril-otro'), ('P2', '25', 'Frecuencia de medidas de seguridad', 'tabla ordinal multicolumna', 'ordinalidad y coherencia por medida'), ('P3', '26', 'Tabaquismo', 'categórica mixta', 'orden parcial, fumador social/pasivo/vapeo/otro'), ('P3', '27', 'Consumo de alcohol', 'categórica nominal/ordinal', 'no/puntual/habitual/otro'), ('P3', '28', 'Bebidas estimulantes', 'tabla ordinal multicolumna', 'nunca/ocasional/frecuente'), ('P3', '29', 'Antidepresivos', 'categórica nominal', 'sí/no/otro; posible texto libre'), ('P3', '30', 'Tranquilizantes', 'categórica nominal', 'sí/no/otro; posible texto libre'), ('P3', '31', 'Tratamiento crónico para el dolor', 'categórica nominal', 'sí/no/otro; posible texto libre'), ('P3', '32', 'Consumo habitual de productos', 'tabla ordinal multicolumna', 'nunca/de vez en cuando/habitualmente'), ('P3', '33', 'Estado de salud general', 'categórica ordinal', 'excelente/bueno/regular/malo/...'), ('P3', '34', 'Enfermedades metabólicas', 'multicolumna ternaria', 'sí/no/no sé por enfermedad'), ('P3', '35', 'Trastornos autoinmunes', 'multicolumna ternaria', 'sí/no/no sé por enfermedad'), ('P3', '36', 'Alergias', 'multicolumna ternaria', 'sí/no/no sé por alergia'), ('P3', '37', 'Enfermedades respiratorias', 'multicolumna ternaria', 'sí/no/no sé por enfermedad'), ('P3', '38', 'Enfermedades cardíacas', 'multicolumna ternaria', 'sí/no/no sé por enfermedad'), ('P3', '39', 'Patologías en los últimos 2 años', 'multicolumna ternaria', 'sí/no/no sé por patología'), ('P3', '40', 'EPOC', 'categórica binaria', 'sí/no, faltantes'), ('P3', '40.1', 'Condiciones de riesgo', 'multirrespuesta', 'cáncer, trasplante, embarazo, no, otro'), ('P4', '41', 'Estado de salud últimos meses', 'categórica ordinal', 'excelente/bueno/regular/malo/otro'), ('P4', '42', 'Patología con ingreso hospitalario', 'categórica nominal', 'sí/no/otro + posible texto'), ('P4', '43', 'Visita a centro hospitalario', 'categórica binaria', 'sí/no'), ('P4', '44', 'Otra patología con asistencia médica', 'categórica nominal', 'sí/no/otro'), ('P4', '45', 'Otra patología sin asistencia médica', 'categórica nominal', 'sí/no/otro'), ('P4', '46', 'Sintomatología reciente', 'multicolumna binaria', 'sí/no por síntoma, consistencia'), ('P5', '47', 'Vacunas y enfermedades previas', 'multicolumna ternaria', 'sí/no/no sé por vacuna/enfermedad'), ('P5', '48', 'Convivencia con positivo COVID', 'categórica nominal', 'sí/no/no sé/otro'), ('P5', '49', 'Sospecha de COVID', 'categórica nominal', 'sí/no/no sé/otro'), ('P5', '50', 'Prueba PCR', 'categórica nominal', 'no/espera/positivo/negativo/otro'), ('P5', '51', 'Test de anticuerpos', 'categórica nominal', 'sí con anticuerpos/sí sin anticuerpos/no/otro')],
    columns=["bloque", "pregunta", "descripcion", "tipo_sugerido", "foco_limpieza"])

display(diccionario_preguntas)

,bloque,pregunta,descripcion,tipo_sugerido,foco_limpieza
0,P1,1,Código postal antes del confinamiento,texto/código,"longitud, dígitos, ceros a la izquierda, faltantes"
1,P1,1.1,Barrio antes del confinamiento,texto nominal,"espacios, mayúsculas, tildes, abreviaturas, duplicados semánticos"
2,P1,2,Sexo,categórica nominal,"H/M, Hombre/Mujer, mayúsculas, categorías extrañas"
3,P1,3,Edad,numérica,"conversión a número, valores imposibles, outliers, imputación"
4,P1,4,Peso (kg),numérica,"conversión a número, unidades, extremos, ceros/imposibles"
5,P1,5,Talla (cm),numérica,"conversión a número, cm frente a m, extremos, ceros/imposibles"
6,P1,6,Tipo de domicilio antes,categórica nominal,"unificar categorías, otros, faltantes"
7,P1,7,Afluencia en la zona antes,categórica ordinal,"bajo/medio/alto, orden lógico, variantes de texto"
8,P1,8,Número de convivientes antes,numérica discreta,"enteros, negativos, extremos, faltantes"
9,P1,9,Edades de convivientes antes,texto/lista,"listas mal formadas, separadores, extracción de resúmenes"


## Construye ahora tu tabla maestra de auditoría

La idea es que tengas **una fila por columna real del XLSX**.  
Después, irás rellenando o ampliando esta tabla durante la práctica.

In [12]:
auditoria = pd.DataFrame({"columna_original": df_trabajo.columns})
auditoria["pregunta"] = auditoria["columna_original"].str.extract(r"^(\d+(?:\.\d+)?)")[0]
auditoria["dtype_raw"] = [df_trabajo[c].dtype for c in df_trabajo.columns]
auditoria["n_missing"] = [df_trabajo[c].isna().sum() for c in df_trabajo.columns]
auditoria["pct_missing"] = [100 * df_trabajo[c].isna().mean() for c in df_trabajo.columns]
auditoria["n_unicos"] = [df_trabajo[c].nunique(dropna=True) for c in df_trabajo.columns]

auditoria = auditoria.merge(diccionario_preguntas, on="pregunta", how="left")

# Columnas que debes completar tú, manualmente o con ayuda de código
auditoria["problemas_detectados"] = ""
auditoria["estrategia_aplicada"] = ""
auditoria["columnas_generadas"] = ""

display(auditoria.head(30))

,columna_original,pregunta,dtype_raw,n_missing,pct_missing,n_unicos,bloque,descripcion,tipo_sugerido,foco_limpieza,problemas_detectados,estrategia_aplicada,columnas_generadas
0,1.CPBARRIO-ANTES_1,1,object,1,0.833333,92,P1,Código postal antes del confinamiento,texto/código,"longitud, dígitos, ceros a la izquierda, faltantes",,,
1,1.CPBARRIO-ANTES_2,1,str,12,10.000000,97,P1,Código postal antes del confinamiento,texto/código,"longitud, dígitos, ceros a la izquierda, faltantes",,,
2,2.SEXO_1,2,str,1,0.833333,2,P1,Sexo,categórica nominal,"H/M, Hombre/Mujer, mayúsculas, categorías extrañas",,,
3,3.EDAD_1,3,object,0,0.000000,46,P1,Edad,numérica,"conversión a número, valores imposibles, outliers, imputación",,,
4,4.PESO_1,4,float64,0,0.000000,37,P1,Peso (kg),numérica,"conversión a número, unidades, extremos, ceros/imposibles",,,
5,5TALLA_1,5,object,0,0.000000,7,P1,Talla (cm),numérica,"conversión a número, cm frente a m, extremos, ceros/imposibles",,,
6,6.DOMICILIO-ANTES_1,6,str,0,0.000000,4,P1,Tipo de domicilio antes,categórica nominal,"unificar categorías, otros, faltantes",,,
7,7.AFLUENCIA-ANTES_1,7,str,41,34.166667,2,P1,Afluencia en la zona antes,categórica ordinal,"bajo/medio/alto, orden lógico, variantes de texto",,,
8,8.CONVIVENCIA-ANTES_1,8,object,0,0.000000,1,P1,Número de convivientes antes,numérica discreta,"enteros, negativos, extremos, faltantes",,,
9,"9.ED,DES-,NTES_1",9,object,80,66.666667,3,P1,Edades de convivientes antes,texto/lista,"listas mal formadas, separadores, extracción de resúmenes",,,


In [13]:
def registrar_auditoria(col, problema, accion, nueva):
    idx = auditoria[auditoria["columna_original"] == col].index[0]
    auditoria.at[idx, "problemas_detectados"] = problema
    auditoria.at[idx, "accion_realizada"] = accion
    auditoria.at[idx, "columna_nueva"] = nueva

# 5. Auditoría global del dataset

Antes de entrar en las preguntas, realiza una auditoría global.

## Tareas

1. Detecta posibles **duplicados exactos**.
2. Detecta valores con espacios iniciales/finales.
3. Busca valores sospechosos repetidos (`"Otro"`, `"No se"`, `"NS/NC"`, `"-"`, etc.).
4. Comprueba qué columnas parecen **numéricas pero están guardadas como texto**.
5. Señala qué columnas parecen:
   - nominales,
   - ordinales,
   - numéricas,
   - multirrespuesta,
   - texto libre.

In [14]:
# TODO: duplicados exactos
# duplicados = ...
# display(duplicados)

duplicados = df_trabajo[df_trabajo.duplicated(keep=False)]
display(duplicados)
print(f"Hay {len(duplicados) // 2} valores duplicados.")

,1.CPBARRIO-ANTES_1,1.CPBARRIO-ANTES_2,2.SEXO_1,3.EDAD_1,4.PESO_1,5TALLA_1,6.DOMICILIO-ANTES_1,7.AFLUENCIA-ANTES_1,8.CONVIVENCIA-ANTES_1,"9.ED,DES-,NTES_1",10.PROFESION-ANTES_1,11.LUGARTRABAJO-ANTES_1,12.OCIO_1,13.TRANSPORTE_1,13.TRANSPORTE_2,13.TRANSPORTE_3,13.TRANSPORTE_4,13.TRANSPORTE_5,13.TRANSPORTE_6,13.TRANSPORTE_7,14.EXTRANJERO_1,15.DESPLAZAMIENTO-NACIONAL_1,16.CPBARRIO-DURANTE_1,16.CPBARRIO-DURANTE_2,17.DOMICILIO-DURANTE_1,18.AFLUENCIA-DURANTE_1,19.CONVIVENCIA-DURANTE_1,20.EDADES-DURANTE_1,21.MASCOTAS_1,21.MASCOTAS_2,21.MASCOTAS_3,21.MASCOTAS_4,21.MASCOTAS_5,22.PROFESION-DURANTE_1,22.PROFESION-DURANTE_2,22.PROFESION-DURANTE_3,22.PROFESION-DURANTE_4,22.PROFESION-DURANTE_5,22.PROFESION-DURANTE_6,22.PROFESION-DURANTE_7,22.PROFESION-DURANTE_8,22.PROFESION-DURANTE_9,22.PROFESION-DURANTE_10,22.PROFESION-DURANTE_11,22.PROFESION-DURANTE_12,23.LUGARTRABAJO-DURANTE_1,24.SEGURIDAD-MES_1,24.SEGURIDAD-MES_2,24.SEGURIDAD-MES_3,24.SEGURIDAD-MES_4,25.SEGURIDAD-FRECUENCIA_1,25.SEGURIDAD-FRECUENCIA_2,25.SEGURIDAD-FRECUENCIA_3,25.SEGURIDAD-FRECUENCIA_4,25.SEGURIDAD-FRECUENCIA_5,26.FUMADOR_1,27.ALCOHOL_1,28.BEBIDAS_1,28.BEBIDAS_2,28.BEBIDAS_3,28.BEBIDAS_4,28.BEBIDAS_5,28.BEBIDAS_6,29.ANTIDEPRESIVOS_1,30.TRANQUILIZANTES_1,31.TRATAMIENTO-DOLOR_1,32.SUPLEMENTOS_1,32.SUPLEMENTOS_2,32.SUPLEMENTOS_3,33.ESTADO-SALUD_1,34.ENF-METABOLICA_1,34.ENF-METABOLICA_2,34.ENF-METABOLICA_3,34.ENF-METABOLICA_4,34.ENF-METABOLICA_5,34.ENF-METABOLICA_6,34.ENF-METABOLICA_7,34.ENF-METABOLICA_8,34.ENF-METABOLICA_9,35.ENF-AUTOINMUNE_1,35.ENF-AUTOINMUNE_2,35.ENF-AUTOINMUNE_3,35.ENF-AUTOINMUNE_4,35.ENF-AUTOINMUNE_5,35.ENF-AUTOINMUNE_6,35.ENF-AUTOINMUNE_7,35.ENF-AUTOINMUNE_8,36.ENF-ALERGIA_1,36.ENF-ALERGIA_2,36.ENF-ALERGIA_3,36.ENF-ALERGIA_4,36.ENF-ALERGIA_5,36.ENF-ALERGIA_6,36.ENF-ALERGIA_7,37.ENF-RESPIRATORIA_1,37.ENF-RESPIRATORIA_2,37.ENF-RESPIRATORIA_3,37.ENF-RESPIRATORIA_4,37.ENF-RESPIRATORIA_5,37.ENF-RESPIRATORIA_6,37.ENF-RESPIRATORIA_7,38.ENF-CARDIACA_1,38.ENF-CARDIACA_2,38.ENF-CARDIACA_3,38.ENF-CARDIACA_4,38.ENF-CARDIACA_5,38.ENF-CARDIACA_6,39.ENF-OTRAS_1,39.ENF-OTRAS_2,39.ENF-OTRAS_3,39.ENF-OTRAS_4,39.ENF-OTRAS_5,39.ENF-OTRAS_6,39.ENF-OTRAS_7,40.EPOC-RIESGO_1,40.EPOC-RIESGO_2,41.SALUD-TRIMESTRE_1,42.HOSPITAL-INGRESO_1,43.HOSPITAL-PRUEBAS_1,44.PATOLOGIA-CONSIS_1,45.PATOLOGIA-SINASIS_1,46.SINTOMATOLOGIA_1,46.SINTOMATOLOGIA_2,46.SINTOMATOLOGIA_3,46.SINTOMATOLOGIA_4,46.SINTOMATOLOGIA_5,46.SINTOMATOLOGIA_6,46.SINTOMATOLOGIA_7,46.SINTOMATOLOGIA_8,46.SINTOMATOLOGIA_9,46.SINTOMATOLOGIA_10,46.SINTOMATOLOGIA_11,46.SINTOMATOLOGIA_12,46.SINTOMATOLOGIA_13,46.SINTOMATOLOGIA_14,46.SINTOMATOLOGIA_15,46.SINTOMATOLOGIA_16,46.SINTOMATOLOGIA_17,46.SINTOMATOLOGIA_18,47.VACUNAS_1,47.VACUNAS_2,47.VACUNAS_3,47.VACUNAS_4,47.VACUNAS_5,47.VACUNAS_6,47.VACUNAS_7,47.VACUNAS_8,47.VACUNAS_9,47.VACUNAS_10,48.COVID-CONTACTO_1,49.COVID-SOSPECHA_1,50.COVID-PCR_1,51.COVID-ANTICUERPOS_1
4736,29011,Virreina,Mujer:,37,54.0,170,Piso en bloque de menos 50 viviendas,Medio (movimiento de residentes),0,NaN,Marketing,NaN,En lugares abiertos y cerrados por igual,Frecuente,Nula,Nula,Nula,Muy baja,Muy baja,Muy baja,No,No he salido de mi comunidad,29011.0,Virreina,Piso en bloque de menos 50 viviendas,Medio (movimiento de residentes),0,NaN,No,NaN,NaN,NaN,NaN,NaN,NaN,Mediante teletrabajo,Mediante teletrabajo,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,En el hogar (teletrabajo/en paro/ tareas domésticas/confinado),Marzo,NaN,NaN,NaN,Muy frecuentemente,Algunas veces,Frecuentemente,Frecuentemente,Nunca,No,NaN,Habitualmente,No consumo,No consumo,No consumo,NaN,NaN,No,No,No,Nunca,De vez en cuando,Nunca,Bueno,No,No,No,No,No,No,No,NaN,NaN,No,No,No,No,No,No,NaN,NaN,NaN,NaN,No,Si,No,No,NaN,No,No,No,No,No,NaN,NaN,No,No,No,No,NaN,NaN,No,No,No,Si,No,NaN,NaN,No,No,Bueno,No,Sí,No,No,No,No,Si,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No,No,No,no
4737,29011,Virreina,Mujer:,37,54.0,170,Piso en bloque de menos 50 viviendas,Medio (movimiento de residentes),0,NaN,Marketing,NaN,En lugares abiertos y 

Hay 4 valores duplicados.


In [15]:
# TODO: ejemplos de búsqueda de valores con espacios o tokens sospechosos
# for c in df_trabajo.columns:
#     ...

cont = 0
tokens = ["Otro", "No se", "NS/NC", "-", "N/A", "Desconocido", "No sé, no me consta", "No se, no me consta", ",", "NA"]

for c in df_trabajo.select_dtypes(["object", "string"]).columns:
    cont += (df_trabajo[c].str.strip() != df_trabajo[c]).sum()

print(f"Hay {cont} valores con espacios iniciales/finales.")
print(f"Los tokens sospechosos los quitamos anteriormente en el apartado 3.")

Hay 5747 valores con espacios iniciales/finales.
Los tokens sospechosos los quitamos anteriormente en el apartado 3.


In [15]:
# TODO: identifica columnas con posible contenido numérico guardado como texto
# pista: intenta convertir con pd.to_numeric(errors="coerce") y compara cuántos valores sobreviven

cols = []
for c in df_trabajo.select_dtypes(["object", "string"]).columns:
    num = pd.to_numeric(df_trabajo[c], errors="coerce")

    if num.notna().any():
        cols.append(c)

print(cols)
print(f"Estas son las columnas que contienen valores numéricos pero no se almacenan como numéricos.")

['1.CPBARRIO-ANTES_1', '1.CPBARRIO-ANTES_2', '3.EDAD_1', '5TALLA_1', '8.CONVIVENCIA-ANTES_1', '9.ED,DES-,NTES_1', '16.CPBARRIO-DURANTE_2', '19.CONVIVENCIA-DURANTE_1', '20.EDADES-DURANTE_1']
Estas son las columnas que contienen valores numéricos pero no se almacenan como numéricos.


# 6. Trabajo obligatorio por bloques del cuestionario

A continuación debes trabajar **bloque a bloque**.  
En cada bloque:

1. Muestra las columnas reales del dataframe que pertenecen a ese bloque.
2. Para **cada columna**:
   - inspecciona valores;
   - detecta problemas;
   - decide una estrategia;
   - escribe el código de limpieza.
3. Actualiza `auditoria`.

**Muy importante:** no basta con decir “esta columna está bien”. Debes demostrarlo con código.

## Bloque P1 — Hábitos de convivencia y sociales **ANTES** del confinamiento

Preguntas que debes revisar en este bloque:

- **1 y 1.1**: código postal y barrio antes del confinamiento.
- **2**: sexo.
- **3, 4, 5**: edad, peso y talla.
- **6 y 7**: tipo de domicilio y afluencia.
- **8 y 9**: número y edades de convivientes.
- **10, 11, 12**: profesión, lugar de trabajo y tiempo libre.
- **13**: transporte habitual.
- **14**: viaje al extranjero.
- **15**: desplazamiento dentro de España.

### Sugerencias específicas

- En códigos postales, cuidado con **ceros a la izquierda**.
- En barrio/ocupación/lugar de trabajo pueden aparecer variantes tipográficas o abreviaturas.
- En edad, peso, talla y convivencia debes buscar **outliers** y valores imposibles.
- En `9` y `13` puede haber respuestas compuestas o listas.
- En `14` puede convenir separar **indicador de viaje** y **destino**.

In [18]:
preguntas_p1 = ["1", "1.1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11", "12", "13", "14", "15"]
cols_p1 = columnas_por_bloque(df_trabajo, preguntas_p1)

display(auditoria[auditoria["columna_original"].isin(cols_p1)])

,columna_original,pregunta,dtype_raw,n_missing,pct_missing,n_unicos,bloque,descripcion,tipo_sugerido,foco_limpieza,problemas_detectados,estrategia_aplicada,columnas_generadas
0,1.CPBARRIO-ANTES_1,1,object,1,0.833333,92,P1,Código postal antes del confinamiento,texto/código,"longitud, dígitos, ceros a la izquierda, faltantes",,,
1,1.CPBARRIO-ANTES_2,1,str,12,10.000000,97,P1,Código postal antes del confinamiento,texto/código,"longitud, dígitos, ceros a la izquierda, faltantes",,,
2,2.SEXO_1,2,str,1,0.833333,2,P1,Sexo,categórica nominal,"H/M, Hombre/Mujer, mayúsculas, categorías extrañas",,,
3,3.EDAD_1,3,object,0,0.000000,46,P1,Edad,numérica,"conversión a número, valores imposibles, outliers, imputación",,,
4,4.PESO_1,4,float64,0,0.000000,37,P1,Peso (kg),numérica,"conversión a número, unidades, extremos, ceros/imposibles",,,
5,5TALLA_1,5,object,0,0.000000,7,P1,Talla (cm),numérica,"conversión a número, cm frente a m, extremos, ceros/imposibles",,,
6,6.DOMICILIO-ANTES_1,6,str,0,0.000000,4,P1,Tipo de domicilio antes,categórica nominal,"unificar categorías, otros, faltantes",,,
7,7.AFLUENCIA-ANTES_1,7,str,41,34.166667,2,P1,Afluencia en la zona antes,categórica ordinal,"bajo/medio/alto, orden lógico, variantes de texto",,,
8,8.CONVIVENCIA-ANTES_1,8,object,0,0.000000,1,P1,Número de convivientes antes,numérica discreta,"enteros, negativos, extremos, faltantes",,,
9,"9.ED,DES-,NTES_1",9,object,80,66.666667,3,P1,Edades de convivientes antes,texto/lista,"listas mal formadas, separadores, extracción de resúmenes",,,


In [40]:
# TODO P1:
# 1) inspecciona una por una las columnas de cols_p1;
# 2) crea columnas limpias cuando sea necesario;
# 3) actualiza auditoria.
#
# Ejemplos orientativos de inspección:
# for c in cols_p1:
#     resumen_columna(df_trabajo, c)

print(f"Primero vamos a ver en general un resumen de cada columna:")
for c in cols_p1:
    resumen_columna(df_trabajo, c, top=120)

Primero vamos a ver en general un resumen de cada columna:


,columna,dtype,n,n_missing,pct_missing,n_unicos
0,1.CPBARRIO-ANTES_1,object,120,1,0.833333,92



Primeros valores no nulos:


4680    14005
4681    28002
4682    29006
4683    38204
4684    38201
4685    29010
4686    29014
4687    14006
4688    29190
4689    26540
4690    46600
4691    25002
4692    29013
4693    29018
4694    41003
4695    28002
4696    29160
4697    35008
4698    08013
4699    29011
4700    29012
4701    29130
4702    29620
4703    16004
4704    29640
4705    29009
4706    29601
4707    29002
4708    28905
4709    41002
4710    41010
4711    13170
4712    46021
4713     1050
4714    19001
4715    41927
4716    14003
4717    51002
4718    15895
4720    01012
4721    29700
4722    29018
4723    29590
4724    29130
4725    29002
4726    14220
4727    29016
4728    29010
4729    29010
4730    11190
4731    29001
4732    29003
4733    29007
4734    08015
4735    14420
4736    29011
4737    29011
4738    06001
4739    29011
4740    25266
4741    29010
4742    03190
4743    13002
4744    29002
4745    29017
4746    28027
4747    23400
4748    24402
4749    13700
4750    45001
4751    29007
4752  


Frecuencias (incluyendo NA):


...

,columna,dtype,n,n_missing,pct_missing,n_unicos
0,15.DESPLAZAMIENTO-NACIONAL_1,str,120,46,38.333333,3



Primeros valores no nulos:


4681    No he salido de mi comunidad
4682                         Córdoba
4683    No he salido de mi comunidad
4684    No he salido de mi comunidad
4686    No he salido de mi comunidad
4687    No he salido de mi comunidad
4689    No he salido de mi comunidad
4690    No he salido de mi comunidad
4692    No he salido de mi comunidad
4693    No he salido de mi comunidad
4695    No he salido de mi comunidad
4696    No he salido de mi comunidad
4698    No he salido de mi comunidad
4699    No he salido de mi comunidad
4700    No he salido de mi comunidad
4701    No he salido de mi comunidad
4702    No he salido de mi comunidad
4704    No he salido de mi comunidad
4705    No he salido de mi comunidad
4706    No he salido de mi comunidad
4707    No he salido de mi comunidad
4710                   Sierra nevada
4711    No he salido de mi comunidad
4715    No he salido de mi comunidad
4716    No he salido de mi comunidad
4718    No he salido de mi comunidad
4720    No he salido de mi comunidad
4


Frecuencias (incluyendo NA):


15.DESPLAZAMIENTO-NACIONAL_1
No he salido de mi comunidad    72
NaN                             46
Córdoba                          1
Sierra nevada                    1
Name: count, dtype: int64

In [53]:
# P1
print(f"---P1---")
print(f"{df_trabajo.loc[df_trabajo["1.CPBARRIO-ANTES_1"].str.len() < 5, "1.CPBARRIO-ANTES_1"]}")
print(f"{df_trabajo.loc[df_trabajo["1.CPBARRIO-ANTES_1"].isnull(), "1.CPBARRIO-ANTES_1"]}")
print("Vemos que tenemos una columna que es el código postal " 
      "y el tipo de columna es un string. A pesar de ser un valor " 
      "numérico, en España se mantiene los 0 a la izquierda, por " 
      "lo que lo mantenemos. Además hay un valor faltante y "
      "también nos aseguraremos que el código postal tenga 5 " 
      "dígitos.")
df_trabajo["1.cp_barrio_antes_1"] = df_trabajo["1.CPBARRIO-ANTES_1"].astype(str).str.extract(r"(\0?\d+)")[0].str.zfill(5)
registrar_auditoria("1.CPBARRIO-ANTES_1",
                    "Valores nulos, el código postal necesita "
                    "5 dígitos, y hay valores que no lo cumplen.",
                    "Crear una nueva columna y rellenar con 0 "
                    "a la izquierda y mas adelante manejaremos los "
                    "nulos",
                    "1.cp_barrio_antes_1")

---P1---
4713    1050
4777    5418
Name: 1.CPBARRIO-ANTES_1, dtype: object
4719    NaN
Name: 1.CPBARRIO-ANTES_1, dtype: object
Vemos que tenemos una columna que es el código postal y el tipo de columna es un string. A pesar de ser un valor numérico, en España se mantiene los 0 a la izquierda, por lo que lo mantenemos. Además hay un valor faltante y también nos aseguraremos que el código postal tenga 5 dígitos.


## Bloque P2 — Hábitos de convivencia y sociales **DURANTE** el confinamiento

Preguntas:

- **16 y 16.1**: código postal y barrio durante el confinamiento.
- **17 y 18**: tipo de domicilio y afluencia.
- **19 y 20**: número y edades de convivientes.
- **21**: mascotas.
- **22**: ámbito y forma de ejercer la profesión.
- **23**: lugar de trabajo.
- **24**: mes de inicio de medidas de seguridad.
- **25**: frecuencia de medidas de seguridad.

### Sugerencias específicas

- Compara P1 frente a P2 cuando tenga sentido: domicilio, afluencia, convivencia, etc.
- En **21** y **22** puede haber problemas de coherencia entre varias marcas.
- En **24** y **25** hay estructura **ordinal**: decide si conviene conservar texto, codificar ordinalmente o ambas cosas.

In [ ]:
preguntas_p2 = ["16", "16.1", "17", "18", "19", "20", "21", "22", "23", "24", "25"]
cols_p2 = columnas_por_bloque(df_trabajo, preguntas_p2)

display(auditoria[auditoria["columna_original"].isin(cols_p2)])

In [ ]:
# TODO P2:
# Analiza y limpia las columnas de cols_p2.
# Para preguntas de tipo ordinal, justifica la codificación elegida.

## Bloque P3 — Estado de salud y hábitos personales

Preguntas:

- **26**: tabaquismo.
- **27**: alcohol.
- **28**: bebidas estimulantes.
- **29, 30, 31**: medicación/tratamientos.
- **32**: consumo de productos.
- **33**: estado de salud general.
- **34, 35, 36, 37, 38, 39**: grupos de enfermedades.
- **40 y 40.1**: EPOC y condiciones de riesgo.

### Sugerencias específicas

- En **26** hay mezcla de categorías que parecen tener orden y otras que no.
- En **28** y **32** hay tablas de frecuencia: trata la ordinalidad con cuidado.
- En **34–39** hay muchas columnas binarias/ternarias: revisa **consistencia** y unifica codificación.
- En **40.1** puede haber multirrespuesta y posible contradicción entre `No` y otra condición marcada.

In [ ]:
preguntas_p3 = ["26", "27", "28", "29", "30", "31", "32", "33", "34", "35", "36", "37", "38", "39", "40", "40.1"]
cols_p3 = columnas_por_bloque(df_trabajo, preguntas_p3)

display(auditoria[auditoria["columna_original"].isin(cols_p3)])

In [ ]:
# TODO P3:
# 1) inspecciona cols_p3;
# 2) crea, si lo consideras útil, variables agregadas:
#    - número de enfermedades metabólicas
#    - número de alergias
#    - número de patologías respiratorias
#    - indicador global de comorbilidad
# 3) documenta todo en auditoria.

## Bloque P4 — Estado de salud durante el confinamiento

Preguntas:

- **41**: estado de salud en los últimos meses.
- **42, 43, 44, 45**: ingreso hospitalario, visitas al hospital y otras patologías.
- **46**: sintomatología.

### Sugerencias específicas

- En **41** hay una variable ordinal clara.
- En **42**, **44** y **45** puede haber mezcla entre sí/no y texto libre en “otro”.
- En **46** tendrás varias columnas binarias: revisa si todas usan la misma codificación y crea variables resumen si están justificadas.

In [ ]:
preguntas_p4 = ["41", "42", "43", "44", "45", "46"]
cols_p4 = columnas_por_bloque(df_trabajo, preguntas_p4)

display(auditoria[auditoria["columna_original"].isin(cols_p4)])

In [ ]:
# TODO P4:
# Limpia cols_p4 y, si procede, crea variables derivadas:
# - numero_sintomas
# - sintoma_respiratorio
# - sintoma_digestivo
# - sintoma_neurologico
# etc.

## Bloque P5 — Información referente a vacunas y COVID

Preguntas:

- **47**: vacunas/enfermedades previas.
- **48**: convivencia con positivo COVID.
- **49**: sospecha de COVID.
- **50**: prueba PCR.
- **51**: anticuerpos.

### Sugerencias específicas

- En **47** vuelve a aparecer estructura multicolumna/ternaria.
- En **50** y **51** conviene revisar si la variable debe quedarse como texto categórico o si puede descomponerse en indicadores más simples.
- Piensa qué variable podría servir como **objetivo** o como **variable de interés clínica**.

In [ ]:
preguntas_p5 = ["47", "48", "49", "50", "51"]
cols_p5 = columnas_por_bloque(df_trabajo, preguntas_p5)

display(auditoria[auditoria["columna_original"].isin(cols_p5)])

In [ ]:
# TODO P5:
# Limpia cols_p5 y decide si conviene crear:
# - covid_pcr_realizada
# - covid_pcr_positiva
# - covid_anticuerpos
# - covid_sospecha_binaria
# etc.

# 7. Tratamiento de valores faltantes

Tras la limpieza básica, debes estudiar **qué columnas tienen faltantes** y decidir, de forma justificada, qué hacer en cada caso.

## Debes incluir ejemplos de:

1. **Eliminación** de filas o columnas, si la justificas.
2. **Imputación simple**:
   - media o mediana para numéricas,
   - moda para categóricas.
3. **Al menos una imputación más elaborada** o una estrategia razonada basada en otras columnas, si el caso lo merece.

## No se acepta

- rellenar todo “a ojo”;
- usar siempre la misma estrategia para todas las columnas;
- imputar sin explicar por qué.

In [ ]:
# TODO:
# crea una tabla con el porcentaje de faltantes y decide qué columnas requieren tratamiento específico
# missing = ...
# display(missing)

In [ ]:
# TODO:
# escribe aquí tus imputaciones justificadas
# ejemplos posibles:
# - imputación de edad por mediana
# - imputación de peso/talla tras detectar outliers
# - imputación de una categoría por moda
# - no imputar y mantener NaN si la ausencia es informativa

# 8. Detección y tratamiento de outliers

Debes aplicar este análisis, al menos, a las variables numéricas que finalmente consideres relevantes.

## Variables candidatas típicas

- edad,
- peso,
- talla,
- número de convivientes,
- resúmenes numéricos que extraigas de otras preguntas,
- IMC.

## Lo que debes hacer

1. Convertir la columna a numérica si es necesario.
2. Visualizar con histogramas y boxplots.
3. Detectar candidatos a outlier con **IQR** o **z-score**.
4. Decidir si son:
   - error de captura,
   - caso extremo real,
   - falta de unidad consistente,
   - dato que debe conservarse.
5. Aplicar una acción justificada:
   - corregir,
   - imputar,
   - marcar,
   - transformar,
   - eliminar (solo si se justifica muy bien).

In [ ]:
# TODO:
# elige aquí tus variables numéricas ya limpiadas y analízalas
# ejemplo:
# ver_numerica(df_trabajo, "3.EDAD_1")

In [ ]:
# TODO:
# implementa al menos una detección de outliers con IQR o z-score

# 9. Transformación de datos

En esta parte debes aplicar **transformaciones con sentido**.

## Obligatorio

### 9.1. Creación de nuevas variables
Debes crear varias variables derivadas. Algunas candidatas son:

- `imc` a partir de peso y talla;
- número de convivientes;
- número de mascotas;
- número de enfermedades por bloque;
- número de síntomas;
- indicadores binarios de riesgo.

### 9.2. Codificación de variables categóricas
Debes usar, cuando corresponda:

- **One-Hot Encoding** para variables nominales;
- **Ordinal Encoding** para variables con orden lógico;
- **Label Encoding** solo si justificas por qué no introduces una jerarquía artificial.

### 9.3. Escalado/normalización
Aplica y compara, al menos sobre algunas variables numéricas limpias:

- Min–Max,
- Z-score,
- o normalización decimal.

### 9.4. Discretización
Discretiza, al menos, una variable continua de forma justificada:
por ejemplo edad, IMC o número de convivientes.

In [ ]:
# TODO:
# crea aquí tus variables derivadas
# Ejemplos posibles:
# df_trabajo["imc"] = ...
# df_trabajo["numero_sintomas"] = ...
# df_trabajo["numero_enfermedades_metabolicas"] = ...

In [ ]:
# TODO:
# aplica aquí codificación de variables categóricas elegidas
# y justifica por qué eliges one-hot, ordinal o label

In [ ]:
# TODO:
# aplica aquí algún escalado a varias variables numéricas limpias
# Puedes crear un dataframe aparte con las variables escaladas

In [ ]:
# TODO:
# discretiza aquí al menos una variable continua
# Ejemplos: edad por rangos, IMC por categorías, etc.

# 10. Reducción de datos

Debes construir una versión del dataset adecuada para un análisis posterior o para un modelo de aprendizaje automático.

## Objetivo

Crear `df_modelo` con aproximadamente **15 variables útiles**.

## Requisitos

1. No basta con elegir 15 columnas “porque sí”.
2. Debes justificar por qué eliminas:
   - columnas redundantes,
   - columnas con demasiados faltantes,
   - columnas excesivamente libres o difíciles de usar,
   - columnas que duplican información.
3. Puedes conservar:
   - columnas limpias originales,
   - variables derivadas,
   - columnas codificadas.
4. Si quieres, puedes apoyar la selección con:
   - sentido del dominio,
   - baja cardinalidad,
   - relación con el objetivo,
   - correlaciones,
   - variabilidad,
   - proporción de faltantes.

In [ ]:
# TODO:
# define aquí df_limpio cuando hayas terminado la limpieza principal
# Sugerencia: usa una selección explícita de columnas limpias
# df_limpio = df_trabajo[[ ... ]].copy()

In [ ]:
# TODO:
# define aquí df_modelo con ~15 variables útiles
# df_modelo = ...

In [ ]:
# TODO:
# justifica aquí tu selección de variables
# Puedes crear una tabla con:
# - variable
# - motivo_para_conservarla
# - tipo
# - tratamiento_aplicado

# 11. Guardado de resultados

Cuando termines, guarda:

- la tabla de auditoría;
- el dataset limpio;
- el dataset reducido.

In [ ]:
# TODO:
# auditoria.to_excel(carpeta / f"Auditoria_EncuestaCOVID_{codigo:02d}.xlsx", index=False)
# df_limpio.to_excel(carpeta / f"DatosEncuestaCOVID_limpio_{codigo:02d}.xlsx", index=False)
# df_modelo.to_excel(carpeta / f"DatosEncuestaCOVID_modelo_{codigo:02d}.xlsx", index=False)

# 12. Checklist final antes de entregar

Comprueba que tu notebook:

- carga solo tus 120 filas;
- mantiene una copia cruda (`df_raw`);
- contiene una auditoría **columna a columna**;
- detecta faltantes, inconsistencias y outliers;
- limpia y transforma con código;
- crea variables derivadas;
- aplica alguna codificación;
- aplica alguna normalización o discretización;
- construye `df_limpio`;
- construye `df_modelo`;
- guarda los resultados.

## Nota importante

El número real de columnas del XLSX debe comprobarse con `df.shape` y `df.info()`.  
No asumas el número de columnas sin verificarlo en tu fichero.